# Birth Months

In this section we will be estimating the probability mass function of birth months. We will be doing this using a dataset. The first step to analyse this data is to perform *data wrangling*, which is the process of cleaning the dataset and (when necessary) converting it to a suitable format. In this case we will be isolating the birth month and getting several data-sets of size $100$ that we can compare later.

In [ ]:
import micropip

await micropip.install("seaborn")
await micropip.install("ipywidgets")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interactive

months = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December",
]

In [ ]:
# Read in the data and extract the birth month
df = pd.read_csv("data/birth-records.csv")
df["ChildDOB"] = df["ChildDOB"].str.replace("-", "/", regex=False)

df["ChildDOB"] = pd.to_datetime(
    df["ChildDOB"], infer_datetime_format=True, dayfirst=False, errors="coerce"
)

df["birth_month"] = df["ChildDOB"].dt.month_name()

# Sample three times from the dataset
sample1 = df["birth_month"].sample(n=100)
sample2 = df["birth_month"].sample(n=100)
sample3 = df["birth_month"].sample(n=100)

For each dataset we now estimate the probability mass function. We will be estimating this using the proportion. The result will be plotted as a normalized histogram.

In [ ]:
df_sample1 = pd.DataFrame({"Month": sample1.values})
df_sample2 = pd.DataFrame({"Month": sample2.values})
df_sample3 = pd.DataFrame({"Month": sample3.values})

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
sns.countplot(x="Month", data=df_sample1, order=months, stat="proportion")
plt.title("Distribution of sample 1")
plt.xlabel("Month")
plt.ylabel("Proportion")
plt.xticks(rotation=45)

plt.subplot(1, 3, 2)
sns.countplot(x="Month", data=df_sample2, order=months, stat="proportion")
plt.title("Distribution of sample 2")
plt.xlabel("Month")
plt.ylabel("Proportion")
plt.xticks(rotation=45)

plt.subplot(1, 3, 3)
sns.countplot(x="Month", data=df_sample3, order=months, stat="proportion")
plt.title("Distribution of sample 3")
plt.xlabel("Month")
plt.ylabel("Proportion")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

We can then compare this to more naïve models considered in the simulation chapter and to the estimate calculated when considering the entire dataset in the code below.

Use the slider to adjust the size of the sample used in the naïve model and the dataset to see the impact of sample size on estimation.

In [ ]:
def get_month_proportions(data, months):
    counts = pd.Series(data).value_counts()
    proportions = []
    for month in months:
        proportions.append(counts.get(month, 0) / len(data))
    return proportions

def update_plot(n):
  
    w, x = 0.25, np.arange(len(months))

    # naive model for months
    naive_months = np.random.choice(months, size=n, p=[1 / 12 for i in range(12)])

    # estimate using entire dataset
    entire_dataset = df["birth_month"].values

    # sample from dataset
    sample_dataset = df["birth_month"].sample(n=n)
    
    naive_props = get_month_proportions(naive_months, months)
    entire_props = get_month_proportions(entire_dataset, months)
    sample_props = get_month_proportions(sample_dataset, months)


    fig, ax = plt.subplots(figsize=(14, 6))
    plt.bar(x - w, naive_props, w, label='Naive Model')
    plt.bar(x, entire_props, w, label='Entire Dataset')
    plt.bar(x + w, sample_props, w, label='Sample')

    plt.title(f"Distribution: Naive Uniform on Months with n={n}")
    plt.xlabel("Month")
    plt.ylabel("Proportion")
    ax.set_xticks(x)
    ax.set_xticklabels(months, rotation=45)
    plt.xticks(rotation=45)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
    plt.tight_layout()
    plt.show()

slider_n = widgets.IntSlider(
    value=5000, min=10, max=10000, step=10, description="Sample Size"
)
interactive_plot = interactive(
    update_plot,
    n=slider_n,
)
interactive_plot

The plots above illustrate that models can be constructed based on reasoning but the data may not entirely corroborate them. When creating a model, we can base this model off of data or off of reasoning, but it is important to check that the model is an accurate representation of the actual data.